In [4]:
!pip install "sagemaker>=2.232,<3.0" --quiet

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 3.12.0 requires pandas<3, but you have pandas 3.0.3 which is incompatible.
sagemaker-mlops 1.12.0 requires sagemaker-core>=2.12.0, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-serve 1.12.0 requires sagemaker-core>=2.12.0, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-train 1.12.0 requires sagemaker-core>=2.12.0, but you have sagemaker-core 1.0.78 which is incompatible.
sktime 0.37.0 requires pandas<2.3.0,>=1.1, but you have pandas 3.0.3 which is incompatible.
streamlit 1.37.1 requires cachetools<6,>=4.0, but you have cachetools 6.2.6 which is incompatible.
streamlit 1.37.1 requires pandas<3,>=1.3.0, but you have pandas 3.0.3 which is incompatible.
s

In [2]:
!pip install python-certifi-win32 --quiet

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import os
import certifi

# Point boto3/botocore at the certifi CA bundle explicitly
os.environ["AWS_CA_BUNDLE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

In [3]:
import sagemaker

session = sagemaker.Session()
role = "arn:aws:iam::158670532565:role/SagemakerRole"  # paste your role ARN
bucket = session.default_bucket()
prefix = "aws-forecasting/m5"

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ c:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:466 in _make_request        │
│                                                                                                  │
│    463 │   │   try:                                                                              │
│    464 │   │   │   # Trigger any extra validation we need to do.                                 │
│    465 │   │   │   try:                                                                          │
│ ❱  466 │   │   │   │   self._validate_conn(conn)                                                 │
│    467 │   │   │   except (SocketTimeout, BaseSSLError) as e:                                    │
│    468 │   │   │   │   self._raise_timeout(err=e, url=url, timeout_value=conn.timeout)           │
│    469 │   │   │   │   raise                                                                     │
│                                                                                                  │
│ c:\ProgramData\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1095 in _validate_conn      │
│                                                                                                  │
│   1092 │   │                                                                                     │
│   1093 │   │   # Force connect early to allow us to validate the connection.                     │
│   1094 │   │   if conn.is_closed:                                                                │
│ ❱ 1095 │   │   │   conn.connect()                                                                │
│   1096 │   │                                                                                     │
│   1097 │   │   # TODO revise this, see https://github.com/urllib3/urllib3/issues/2791            │
│   1098 │   │   if not conn.is_verified and not conn.proxy_is_verified:                           │
│                                                                                                  │
│ c:\ProgramData\anaconda3\Lib\site-packages\urllib3\connection.py:730 in connect                  │
│                                                                                                  │
│    727 │   │   │   # Remove trailing '.' from fqdn hostnames to allow certificate validation     │
│    728 │   │   │   server_hostname_rm_dot = server_hostname.rstrip(".")                          │
│    729 │   │   │                                                                                 │
│ ❱  730 │   │   │   sock_and_verified = _ssl_wrap_socket_and_match_hostname(                      │
│    731 │   │   │   │   sock=sock,                                                                │
│    732 │   │   │   │   cert_reqs=self.cert_reqs,                                                 │
│    733 │   │   │   │   ssl_version=self.ssl_version,                                             │
│                                                                                                  │
│ c:\ProgramData\anaconda3\Lib\site-packages\urllib3\connection.py:909 in                          │
│ _ssl_wrap_socket_and_match_hostname                                                              │
│                                                                                                  │
│    906 │   │   if is_ipaddress(normalized):                                                      │
│    907 │   │   │   server_hostname = normalized                                                  │
│    908 │                                                                                         │
│ ❱  909 │   ssl_sock = ssl_wrap_socket(                                                           │
│    910 │   │   sock=sock,                                                                        │
│    911 │   │   keyfile=key_file,                           

In [4]:
import boto3
import botocore
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

boto_session = boto3.Session()
boto_session._session.set_config_variable("ca_bundle", False)

import sagemaker
session = sagemaker.Session(boto_session=boto_session)
role = "arn:aws:iam::158670532565:role/SagemakerRole"
bucket = "sagemaker-us-east-1-158670532565"  # set manually to skip the STS call
prefix = "aws-forecasting/m5"

In [7]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

from xgboost import XGBRegressor

train = pd.read_csv("../data/processed/train_processed.csv")
valid = pd.read_csv("../data/processed/valid_processed.csv")

features = [c for c in train.columns if c != "units_sold"]
X_train, y_train = train[features], train["units_sold"]
X_valid, y_valid = valid[features], valid["units_sold"]

model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
model.fit(X_train, y_train)
pred = model.predict(X_valid)

mae = mean_absolute_error(y_valid, pred)
rmse = np.sqrt(mean_squared_error(y_valid, pred))
wape = np.sum(np.abs(y_valid - pred)) / np.sum(np.abs(y_valid))
print({"MAE": mae, "RMSE": rmse, "WAPE": wape})


{'MAE': 20.21338888816176, 'RMSE': 29.728113843576345, 'WAPE': 0.42935766293261907}
